# TP3 - Detección del logo Coca-Cola

**Visión por Computadora**

El objetivo es encontrar el logotipo de Coca-Cola en las imágenes provistas utilizando el template dado.

La solución se basa únicamente en herramientas trabajadas en clase:

- Template Matching.
- Canny como preprocesamiento.
- cambio de escala con `cv.resize`;
- umbral sobre el mapa de coincidencia;
- IoU para evitar detecciones repetidas.

La exploración de métodos y parámetros quedó separada en otro notebook.


In [ ]:
import os
import glob
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


## 1. Datos


In [ ]:
base_dir = '.'

if not os.path.exists(os.path.join(base_dir, 'images')):
    base_dir = '/mnt/data/tp3data'

images_dir = os.path.join(base_dir, 'images')
template_path = os.path.join(base_dir, 'template', 'pattern.png')

image_paths = sorted(glob.glob(os.path.join(images_dir, '*')))

template_color = cv.imread(template_path)
template_gray = cv.cvtColor(template_color, cv.COLOR_BGR2GRAY)

print('Imágenes encontradas:', len(image_paths))


In [ ]:
plt.figure(figsize=(7, 3))
plt.imshow(cv.cvtColor(template_color, cv.COLOR_BGR2RGB))
plt.title('Template utilizado')
plt.axis('off')
plt.show()


## 2. Funciones auxiliares

### Bordes

Uso Canny tanto sobre la imagen como sobre el template.

Los umbrales se obtienen a partir de la mediana de intensidad para no definir un valor diferente para cada imagen.


In [ ]:
def canny_simple(gray):
    mediana = np.median(gray)

    low = int(max(0, 0.67 * mediana))
    high = int(min(255, 1.33 * mediana))

    return cv.Canny(gray, low, high)


### Intersection over Union (IoU)

La uso para decidir si dos bounding boxes corresponden prácticamente a la misma detección.


In [ ]:
def iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    union = area_a + area_b - inter

    if union == 0:
        return 0

    return inter / union


In [ ]:
def nms_simple(detecciones, umbral_iou=0.30):
    detecciones = sorted(
        detecciones,
        key=lambda d: d['score'],
        reverse=True
    )

    seleccionadas = []

    for det in detecciones:
        repetida = False

        for elegida in seleccionadas:
            if iou(det['box'], elegida['box']) > umbral_iou:
                repetida = True
                break

        if not repetida:
            seleccionadas.append(det)

    return seleccionadas


## 3. Template Matching multiescala

El tamaño del logo cambia entre imágenes.

Para resolverlo genero varias versiones del mismo template y aplico `TM_CCOEFF_NORMED` en cada escala.


In [ ]:
scales = np.linspace(0.10, 1.60, 40)


In [ ]:
def detectar_logo(
    imagen,
    template_gray,
    proporcion_umbral=0.82,
    umbral_iou=0.30
):
    gray = cv.cvtColor(imagen, cv.COLOR_BGR2GRAY)

    imagen_edges = canny_simple(gray)
    template_edges = canny_simple(template_gray)

    candidatos = []

    for scale in scales:
        temp = cv.resize(
            template_edges,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv.INTER_AREA
        )

        h, w = temp.shape

        if h < 8 or w < 8:
            continue

        if h >= imagen_edges.shape[0] or w >= imagen_edges.shape[1]:
            continue

        respuesta = cv.matchTemplate(
            imagen_edges,
            temp,
            cv.TM_CCOEFF_NORMED
        )

        max_val = float(respuesta.max())

        # Umbral relativo al mejor valor de la escala actual
        threshold = proporcion_umbral * max_val

        ys, xs = np.where(respuesta >= threshold)

        for x, y in zip(xs, ys):
            candidatos.append({
                'box': (x, y, x + w, y + h),
                'score': float(respuesta[y, x]),
                'scale': float(scale)
            })

    detecciones = nms_simple(
        candidatos,
        umbral_iou=umbral_iou
    )

    return detecciones


In [ ]:
def dibujar_detecciones(imagen, detecciones, max_detecciones=None):
    salida = imagen.copy()

    if max_detecciones is not None:
        detecciones = detecciones[:max_detecciones]

    for det in detecciones:
        x1, y1, x2, y2 = det['box']

        cv.rectangle(
            salida,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv.putText(
            salida,
            f"{det['score']:.2f}",
            (x1, max(15, y1 - 5)),
            cv.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 255, 0),
            1
        )

    return salida


# Ítem 1 - Una detección por imagen sin falsos positivos

Para este punto tomo la detección con mayor score de cada imagen.

De esa forma el algoritmo devuelve un único bounding box por imagen.


In [ ]:
resultados_item1 = {}

for path in image_paths:
    nombre = os.path.basename(path)

    if nombre == 'coca_multi.png':
        continue

    imagen = cv.imread(path)

    detecciones = detectar_logo(
        imagen,
        template_gray
    )

    if len(detecciones) > 0:
        resultados_item1[nombre] = detecciones[0]
    else:
        resultados_item1[nombre] = None

for nombre, det in resultados_item1.items():
    if det is None:
        print(nombre, '-> sin detección')
    else:
        print(
            nombre,
            '-> score:',
            round(det['score'], 3),
            '- escala:',
            round(det['scale'], 3)
        )


In [ ]:
cantidad = len(resultados_item1)

fig, axes = plt.subplots(
    int(np.ceil(cantidad / 2)),
    2,
    figsize=(14, 5 * int(np.ceil(cantidad / 2)))
)

axes = np.array(axes).ravel()

for ax, (nombre, det) in zip(axes, resultados_item1.items()):
    path = os.path.join(images_dir, nombre)
    imagen = cv.imread(path)

    if det is not None:
        salida = dibujar_detecciones(
            imagen,
            [det]
        )
    else:
        salida = imagen

    ax.imshow(cv.cvtColor(salida, cv.COLOR_BGR2RGB))
    ax.set_title(nombre)
    ax.axis('off')

for ax in axes[len(resultados_item1):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


# Ítem 2 - Múltiples detecciones en `coca_multi.png`

En esta imagen no tomo solamente el máximo.

Conservo las coincidencias que superan el umbral relativo y después aplico IoU para evitar que un mismo logo quede marcado muchas veces.


In [ ]:
multi_path = os.path.join(images_dir, 'coca_multi.png')

multi = cv.imread(multi_path)

detecciones_multi = detectar_logo(
    multi,
    template_gray
)

print('Detecciones luego de NMS:', len(detecciones_multi))


In [ ]:
# Muestro las detecciones de mayor confianza.
# Si hay muchas respuestas residuales, este número permite
# concentrar la visualización en las principales.

cantidad_a_mostrar = min(20, len(detecciones_multi))

salida_multi = dibujar_detecciones(
    multi,
    detecciones_multi,
    max_detecciones=cantidad_a_mostrar
)

plt.figure(figsize=(16, 9))
plt.imshow(cv.cvtColor(salida_multi, cv.COLOR_BGR2RGB))
plt.title(
    'coca_multi.png - '
    + str(cantidad_a_mostrar)
    + ' detecciones de mayor score'
)
plt.axis('off')
plt.show()


# Ítem 3 - Generalización a todas las imágenes

Finalmente aplico exactamente la misma función a todas las imágenes.

No cambio el método según el archivo.


In [ ]:
resultados_generales = {}

for path in image_paths:
    nombre = os.path.basename(path)
    imagen = cv.imread(path)

    detecciones = detectar_logo(
        imagen,
        template_gray
    )

    resultados_generales[nombre] = detecciones

    print(
        nombre,
        '->',
        len(detecciones),
        'detecciones después de NMS'
    )


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for i, path in enumerate(image_paths):
    nombre = os.path.basename(path)
    imagen = cv.imread(path)

    detecciones = resultados_generales[nombre]

    # En las imágenes simples interesa el máximo.
    # En coca_multi se muestran varias detecciones.
    if nombre == 'coca_multi.png':
        mostrar = detecciones[:20]
    else:
        mostrar = detecciones[:1]

    salida = dibujar_detecciones(
        imagen,
        mostrar
    )

    axes[i].imshow(cv.cvtColor(salida, cv.COLOR_BGR2RGB))
    axes[i].set_title(nombre)
    axes[i].axis('off')

for j in range(len(image_paths), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


## Conclusiones

La solución utiliza Template Matching como método principal.

El cambio de escala permite buscar el mismo patrón cuando el logo aparece con tamaños diferentes. El uso de Canny reduce la influencia de diferencias de color o iluminación y concentra la comparación en los contornos.

Para múltiples detecciones no alcanza con tomar solamente el máximo del mapa de respuesta, por lo que se utiliza un umbral y luego IoU para eliminar bounding boxes repetidos.

El mismo procedimiento se aplica a todas las imágenes, sin definir una lógica distinta para cada archivo.
